# Part 3a: Reuse Factor

In Part 1c we used the default hls4ml configuration, which maps the entire neural network onto the FPGA with maximum parallelism, so that all multiply-accumulate operation run in parallel. This gives the lowest possible latency, but consumes a lot of resources.

The **reuse factor (RF)** is a variable in hls4ml for tuning the trade-off between resource consumption and latency. In this notebook we explore how changing it affects the synthesized design.

Run either `1_getting_started/1a_train_keras.ipynb` or `1_getting_started/1b_train_pytorch.ipynb` first, then set `MODEL_TYPE` below.

In [ ]:
MODEL_TYPE = 'keras'  # set to 'pytorch' if you used the PyTorch notebook in Part 1

os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

In [ ]:
import os

os.environ['KERAS_BACKEND'] = 'tensorflow'

import numpy as np
import matplotlib.pyplot as plt
import sys

sys.path.append('..')
import plotting
import hls4ml
from sklearn.metrics import accuracy_score

%matplotlib inline

# Load the data
X_test = np.ascontiguousarray(np.load('../data/jet-tagging/X_test.npy'), dtype=np.float32)
y_test = np.load('../data/jet-tagging/y_test.npy')
classes = np.load('../data/jet-tagging/classes.npy', allow_pickle=True)

## Load the trained model

Load the model saved by Part 1a or 1b.

In [ ]:
if MODEL_TYPE == 'keras':
    import tensorflow as tf

    trained_model = tf.keras.models.load_model('../models/keras_model_part1.h5')
    y_ref = trained_model.predict(X_test)

elif MODEL_TYPE == 'pytorch':
    import torch
    import torch.nn as nn

    class JetTagger(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(16, 64)
            self.fc2 = nn.Linear(64, 32)
            self.fc3 = nn.Linear(32, 32)
            self.output = nn.Linear(32, 5)

        def forward(self, x):
            x = torch.relu(self.fc1(x))
            x = torch.relu(self.fc2(x))
            x = torch.relu(self.fc3(x))
            return torch.softmax(self.output(x), dim=1)

    trained_model = JetTagger()
    trained_model.load_state_dict(torch.load('../models/pytorch_weights_part1.pt'))
    trained_model.eval()
    with torch.no_grad():
        y_ref = trained_model(torch.FloatTensor(X_test)).numpy()

print('Accuracy: {:.4f}'.format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_ref, axis=1))))

## What is the ReuseFactor?

In the default (`ReuseFactor = 1`) configuration, hls4ml instantiates one multiplier for every weight in the network. All multiplications for a given layer happen in a single clock cycle, giving the minimum possible latency — but using the most multipliers.

Setting `ReuseFactor = N` tells hls4ml to time-multiplex the same multiplier hardware across `N` weight-input pairs. This means the layer takes `N` clock cycles to compute instead of one, but uses roughly `1/N` as many multipliers.

![Reuse factor diagram](../images/part3a_reuse_factor.png)

The reuse factor must evenly divide the number of weights in each layer. For example, the first layer has `16 × 64 = 1024` weights, so valid reuse factors include 1, 2, 4, 8, 16, 32, 64, etc.

Changing the reuse factor does **not** change the model accuracy — the same arithmetic is performed, just spread over more clock cycles. We will verify this below.

## Set ReuseFactor = 4

Let's create a new configuration with `ReuseFactor = 4` set globally. Note that we use `granularity='model'` here, which applies one set of defaults to all layers — equivalent to the config from Part 1c.

In [ ]:
if MODEL_TYPE == 'keras':
    config = hls4ml.utils.config_from_keras_model(trained_model, granularity='model', backend='Vitis')
elif MODEL_TYPE == 'pytorch':
    config = hls4ml.utils.config_from_pytorch_model(trained_model, input_shape=(16,), granularity='model', backend='Vitis')

config['Model']['ReuseFactor'] = 4
print('-----------------------------------')
plotting.print_dict(config)
print('-----------------------------------')

In [ ]:
if MODEL_TYPE == 'keras':
    hls_model = hls4ml.converters.convert_from_keras_model(
        trained_model,
        hls_config=config,
        backend='Vitis',
        output_dir='../hls4ml_prjs_/hls4ml_prj_reuse_part3a',
        part='xcu250-figd2104-2L-e',
    )
elif MODEL_TYPE == 'pytorch':
    hls_model = hls4ml.converters.convert_from_pytorch_model(
        trained_model,
        hls_config=config,
        backend='Vitis',
        output_dir='../hls4ml_prjs_/hls4ml_prj_reuse_part3a',
        part='xcu250-figd2104-2L-e',
    )

hls_model.compile()
y_hls = hls_model.predict(X_test)

## Compare

Changing the reuse factor only affects resource usage and latency — not accuracy. Let's verify that the accuracy and ROC curves are identical.

In [ ]:
print('{} Accuracy: {:.4f}'.format(MODEL_TYPE, accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_ref, axis=1))))
print('hls4ml Accuracy: {:.4f}'.format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls, axis=1))))

fig, ax = plt.subplots(figsize=(9, 9))
_ = plotting.makeRoc(y_test, y_ref, list(classes))
plt.gca().set_prop_cycle(None)
_ = plotting.makeRoc(y_test, y_hls, list(classes), linestyle='--')

from matplotlib.lines import Line2D
from matplotlib.legend import Legend

lines = [Line2D([0], [0], ls='-'), Line2D([0], [0], ls='--')]
leg = Legend(ax, lines, labels=[MODEL_TYPE, 'hls4ml (RF=4)'], loc='lower right', frameon=False)
ax.add_artist(leg)

## Synthesize

Now let's synthesize with `ReuseFactor = 4` and compare the resource report against the Part 1c baseline (`ReuseFactor = 1`) to see the effect.

**This can take several minutes.**

In [ ]:
hls_model.build(csim=False)

## Compare reports

Print both reports and compare the DSP and LUT usage. With `ReuseFactor = 4`, you should see roughly a quarter as many DSPs as in the Part 1c baseline, at the cost of approximately four times the latency (in clock cycles).

In [ ]:
print('ReuseFactor = 4:')
hls4ml.report.read_vivado_report('../hls4ml_prjs_/hls4ml_prj_reuse_part3a')

In [ ]:
print('ReuseFactor = 1 (Part 1c baseline):')
hls4ml.report.read_vivado_report('../hls4ml_prjs_/hls4ml_prj_base_part1')

## Further reading

For more details, see: Schulte, Ramhorst, Sun et al., "hls4ml: A Flexible, Open-Source Platform for Deep Learning Acceleration on Reconfigurable Hardware", ACM Trans. Reconfigurable Technol. Syst. (2026), [doi:10.1145/3801979](https://dl.acm.org/doi/abs/10.1145/3801979)